# 📘 07 · Deduplicar IMPORTADORES por RAZÓN SOCIAL + PAÍS

Toma una base de importadores —una fila por `(razón social, país)`— y devuelve una
**correlativa**: cada nombre y país originales junto al **nombre final** y el **país
final** que se les asignó, más la evidencia para juzgar si ese emparejamiento es creíble.

**Variables de empalme:** razón social y país. Dos registros solo se unen si el país
canónico coincide **exactamente** y el nombre supera el umbral. El país es además
bloqueo duro: registros de países distintos ni siquiera se comparan.

---

## 🧭 Mapa del notebook

Dos tipos de celda, a propósito: las de **definición** (se corren una vez y no se editan)
y las de **configuración/ejecución** (cortas, es lo único que usted toca).

- 🔌 **1 · Entorno** *(definición)* — monta Drive, **verifica que el árbol del paquete esté completo** y carga `rues-linker` desde Drive. Una vez por sesión.
- 🧰 **2 · Extras** *(definición)* — todas las funciones auxiliares. Una vez por sesión.
- 🎛️ **3 · Configuración** ← **LA CELDA QUE USTED EDITA**: rutas, perfil de columnas y parámetros del motor.
- 🌍 **4 · Catálogo de países** *(opcional)* — solo si su base trae grafías que la librería no conoce.
- 🧹 **5 · Listas de limpieza** *(opcional)* — sufijos legales y genéricos propios de su dominio.
- 📥 **6 · Lectura y perfilado** — lee, convierte métricas midiendo lo que se pierde, perfila, **comprueba la cobertura del catálogo de países** y corre un smoke test.
- ▶️ **7 · Ejecución** — el motor, **fase por fase, cronometrado y narrando el avance**.
- 🔍 **8 · Control de calidad** — invariantes, métricas, sensibilidad, recall del bloqueo y muestra a revisar.
- 💾 **9 · Exportación** — Parquet primero (checkpoint), luego XLSX, `metadata.json` y copia a Drive.
- 📖 **10 · Referencia** — cómo leer el resultado, cómo calibrar y qué NO hace este notebook.

## 📍 Rutas institucionales de esta configuración

- **Código (paquete descomprimido):** `/content/drive/MyDrive/ProColombia/rues_linker_pruebas`
- **Datos de entrada:** `/content/drive/MyDrive/ProColombia/0A. Datos/Importadores`
- **Resultados:** `/content/drive/MyDrive/ProColombia/1B. Resultados/Rues-Linker/Importadores`

## 🧾 Contrato de entrada vigente

Perfil `snowflake_v2`, correspondiente a esta consulta:

```sql
SELECT
    RAZON_SOCIAL_IMPORTADOR,
    PAIS_ESTANDAR,
    SUM(USD_FOB) AS USD_FOB_TOTAL,
    COUNT(*)     AS NUM_REGISTROS
FROM DWH_PROCOLOMBIA_PRUEBAS.GOLD.VW_EXPORTACIONES_COLOMBIANAS_LECTURA_V2
WHERE FECHA BETWEEN 201601 AND 202607
  AND RAZON_SOCIAL_IMPORTADOR IS NOT NULL
GROUP BY RAZON_SOCIAL_IMPORTADOR, PAIS_ESTANDAR
ORDER BY USD_FOB_TOTAL DESC;
```

Los nombres de columna se cambian en **un solo lugar**: el diccionario `PERFILES_ENTRADA`
de la celda 3. Para otra vista, añada un perfil y cambie la variable `PERFIL`.

> ⚠️ `RAZON_SOCIAL_IMPORTADOR IS NOT NULL` **no** filtra cadenas vacías ni centinelas
> (`'0'`, `'N/A'`, `'NO DISPONIBLE'`, `'A LA ORDEN'`). En esta base son 271 filas con el
> **31 % del valor FOB**: `NO DISPONIBLE` solo, 69 filas —una por país— y 21,7 %. Cada
> una queda en su propio grupo, marcada `sin_nombre_utilizable`, y la celda 6 las cuenta
> antes de correr nada. La lista es la del motor (`PLACEHOLDERS`), no una copia.

## 🌍 Países que el catálogo no reconoce (celda 6)

`PAIS_ESTANDAR` viene estandarizado desde Snowflake, pero «estandarizado» no significa
«en el catálogo»: la primera corrida sobre `snowflake_v2` se detuvo porque trajo grafías
que `record_linkage.paises` no conoce. El país es el **bloqueo duro**, así que una grafía
sin clasificar no es un detalle cosmético: medido, la misma razón social bajo tres
grafías sin clasificar distintas se fusionaba en un solo importador.

La celda 6 lo comprueba **sobre la base completa y antes del smoke test**, con un mapeo
vectorizado que cuesta milisegundos, y aplica `PAISES_SIN_CLASIFICAR` (celda 3):

- **`detener`** *(defecto)* — se detiene y lista cada grafía, cuántas filas trae y el país
  del catálogo más parecido. Usted decide, en la celda 4: si **es** un país, alias en
  `GRAFIAS_ADICIONALES`; si **no** lo es y no tiene sentido catalogarlo (`OTROS`,
  `NO DEFINIDO`), lo declara en `NO_SON_PAISES` — se aísla con su propio `PAIS_FINAL` y
  la corrida sigue, pero **cualquier otra grafía nueva seguirá deteniéndose**. Así se
  acepta lo conocido sin apagar la guardia.
- **`aislar`** — todo lo que no esté en el catálogo se aísla, declarado o no. Útil para
  explorar una base nueva; **no** para producción, porque deja de avisar. Una invariante
  lo **mide** en ambos modos: «ningún grupo mezcla dos grafías de país».

La primera corrida real sobre `snowflake_v2` trajo 19 grafías: 18 eran países que
faltaban en el catálogo —Irán, Afganistán, Liechtenstein, Vanuatu, Samoa…— y ya están
(0.22.4); la otra era `OTROS`, 444 filas, que esta configuración declara en
`NO_SON_PAISES`.

La sugerencia del catálogo es una **ayuda, no una regla**: `GUINEA` y `GUINEA-BISSAU` se
parecen y son países distintos.

## 🌳 Verificación del árbol de Drive (celda 1)

El modo de falla característico de FUSE no es que Drive no monte: es que monte a
medias. La carpeta se ve completa en el explorador y sin embargo falta un `.py`, o
llega con 0 bytes. Eso no se nota al importar `record_linkage` —que solo carga una
parte del árbol— y reaparece como `ModuleNotFoundError` a los siete minutos de
cómputo, con la corrida perdida.

La celda 1 lo cierra con dos comprobaciones distintas, porque miran cosas distintas:

- **Inventario en disco** — los **18 subpaquetes** y los **146 módulos** del árbol de
  0.22.4 deben estar, y ninguno puede pesar 0 bytes. Las cifras están medidas sobre el
  paquete de referencia y atadas a la versión, que la misma celda verifica.
- **Importación exhaustiva** — se importan *todos* los módulos con `pkgutil.walk_packages`.
  Un archivo a medio sincronizar existe y pesa más de 0 bytes, así que la única forma de
  saber que está sano es cargarlo. Los 6 módulos que dependen de extras (`optuna`,
  `plotly`, `matplotlib`, `seaborn`) se reportan como omitidos, no como falla.

Coste: menos de un segundo el primero, unos pocos el segundo. Ambos corren antes de leer
un solo dato.

## ⏱️ Cronómetro por fase

La fachada `deduplicar_importadores` encadena ocho etapas en una sola llamada, así que
desde afuera solo se puede medir el total. Con `USAR_FASES = True` el notebook ejecuta
esas mismas ocho etapas explícitamente y cronometra cada una, más la lectura, el control
de calidad y la exportación. Al final imprime el reparto con barras y exporta la hoja
**TIEMPOS**.

Que la ruta por fases sea idéntica a la fachada no se afirma: el smoke test corre **las
dos** sobre la muestra y compara `CORRELATIVA` y `GOLDEN`. Si difieren, avisa y usted
puede volver a la fachada con `USAR_FASES = False`.

Durante la fase larga el motor narra su avance (`MOSTRAR_AVANCE = True`): filas leídas,
cobertura de países, tamaño del vocabulario y un contador de particiones que se
actualiza sobre la misma línea. Antes de esa fase se imprime el **panorama**: cuántas
particiones hay, cuántos nombres únicos y cuáles son las más grandes — que es donde se
va el tiempo.

## ⚠️ Política anti-FUSE (por qué hay copias locales)

Leer y escribir intensivamente sobre el montaje de Drive es la causa más común de que una
corrida larga en Colab muera con *«transport endpoint is not connected»*. Por eso:

- la **entrada** se copia una vez de Drive al disco local de la VM (`preparar_insumo_local`);
- la **salida** se escribe en un directorio local determinista (`dir_trabajo_seguro`) y **luego** se copia a Drive;
- el **código** sí se importa directamente de Drive: se lee una sola vez, al importar.

## 🧩 Dónde vive la lógica

En `record_linkage`, no en estas celdas. El motor —comparador, canonizador de países,
cobertura por estrellas, invariantes— vive en:

- Comparador de razones sociales sin NIT → `matching.nombre_idf`
- Términos no distintivos, declarados → `matching.genericos`
- Catálogo ISO 3166 + canonizador → `paises`
- Cobertura por estrellas → `engine.cobertura`
- Saneamiento (HTML, mojibake, siglas) → `processing.saneamiento`
- El flujo completo y su control de calidad → `flujo.importadores`

Este notebook **configura y audita**. Si algo del motor le parece mal, se corrige en la
librería y se mide con el banco — no se parchea en una celda.

## 🚨 Los cuatro modos de falla que el motor ya cierra

- **Imán genérico** — `INTERNATIONAL` agrupó 157 razones sociales; `MQE`, 214. Se ataca exigiendo que se comparta un token no genérico y distintivo.
- **Prefijo compartido** — `COMERCIALIZADORA ATLANTA C.A.` vs `…ATLANTIC C.A.` → JW 0,972. Se ataca con ponderación IDF: decide el token informativo, no el prefijo.
- **Encadenamiento** — grupos de 200 empresas distintas. Se ataca con cobertura por estrellas: todo miembro a ≤ umbral de SU nombre final.
- **País por parecido** — `TURQUÍA`/`TÜRKIYE` no se parecen y son el mismo país; `GUINEA`/`GUINEA-BISSAU` se parecen y son distintos. Se ataca con catálogo declarado, no con *fuzzy matching*.

> Motor: **rues-linker 0.22.4**.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
#  🔌 1 · ENTORNO — montar Drive y cargar rues-linker DESDE Drive
#  Celda de DEFINICIÓN. Ejecutar UNA vez por sesión; normalmente no se edita.
# ══════════════════════════════════════════════════════════════════════════
from __future__ import annotations

import importlib
import subprocess
import sys
from importlib.metadata import PackageNotFoundError
from importlib.metadata import version as version_de_distribucion
from pathlib import Path

try:
    import tomllib  # Python ≥ 3.11
except ModuleNotFoundError:  # Python 3.10
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tomli"], check=True)
    import tomli as tomllib  # type: ignore[no-redef]

#  Carpeta de Drive donde usted descomprime el paquete. Ahí solo va CÓDIGO.
RUTA_PAQUETE_DRIVE = "/content/drive/MyDrive/ProColombia/rues_linker_pruebas"  # @param {type:"string"}
VERSION_OBJETIVO = "0.22.4"

#  ── Inventario del árbol de rues-linker 0.22.4 ─────────────────────────────
#  Medido sobre el paquete de referencia, no supuesto:
#    find src/record_linkage -type d -exec test -f {}/__init__.py \; -print   → 18
#    find src/record_linkage -name '*.py' ! -path '*__pycache__*' | wc -l      → 146
#  Sirve para detectar una sincronización de Drive a medias. Está atado a la
#  versión: la celda se detiene si el paquete no es exactamente 0.22.4.
SUBPAQUETES_ESPERADOS: tuple[str, ...] = (
    "classifier", "config", "deduplication", "engine", "engine.lsh", "evaluation",
    "exporters", "flujo", "golden", "ingestion", "matching", "optimization",
    "paises", "pipeline", "processing", "reporting", "testing", "utils",
)
MODULOS_ESPERADOS = 146

#  Los 6 módulos que dependen de un extra no instalado por defecto. Medido con:
#    grep -rln "^import optuna\|^import plotly\|^import matplotlib\|^import seaborn"
EXTRAS_TOLERADOS: frozenset[str] = frozenset(
    {"optuna", "plotly", "matplotlib", "seaborn", "upsetplot", "snowflake"}
)


def montar_drive() -> Path:
    """Monta Google Drive si aún no lo está y devuelve la raíz de MyDrive."""
    raiz = Path("/content/drive/MyDrive")
    if raiz.is_dir():
        print("✅ Drive ya montado")
        return raiz
    try:
        from google.colab import drive  # type: ignore[import-not-found]
    except ModuleNotFoundError as exc:
        raise RuntimeError(
            "Qué pasó: no se encontró `google.colab`.\n"
            "Por qué importa: este notebook lee el código Y los datos desde Drive.\n"
            "Qué hacer: ábralo en Google Colab, o reemplace las rutas por las de su máquina."
        ) from exc
    drive.mount("/content/drive")
    if not raiz.is_dir():
        raise RuntimeError(
            "Qué pasó: el montaje terminó pero no existe /content/drive/MyDrive.\n"
            "Qué hacer: reintente el montaje y autorice la cuenta de Drive correcta."
        )
    print("✅ Drive montado")
    return raiz


def raiz_del_paquete(carpeta: str | Path) -> Path:
    """Ubica la carpeta que contiene `src/record_linkage`.

    Tolera los dos casos: que usted haya subido el CONTENIDO del .zip, o la
    carpeta que el .zip crea al descomprimirse (p. ej. `rues_linker_pruebas`).
    """
    carpeta = Path(carpeta)
    if not carpeta.is_dir():
        raise FileNotFoundError(
            f"Qué pasó: no existe la carpeta del paquete {carpeta}.\n"
            "Por qué importa: es de donde se carga el motor.\n"
            "Qué hacer: descomprima el .zip en esa ruta de Drive y revise el nombre exacto."
        )
    for cand in [carpeta, *sorted(p for p in carpeta.iterdir() if p.is_dir())]:
        if (cand / "src" / "record_linkage" / "__init__.py").is_file():
            return cand
    visto = ", ".join(sorted(p.name for p in carpeta.iterdir())[:15]) or "(vacía)"
    raise FileNotFoundError(
        f"Qué pasó: bajo {carpeta} no aparece `src/record_linkage/__init__.py`.\n"
        "Por qué importa: sin el código fuente no hay motor.\n"
        f"Qué hacer: la carpeta contiene: {visto}. Descomprima el .zip completo allí."
    )


def verificar_arbol(raiz: Path) -> dict[str, int]:
    """Comprueba que el árbol del paquete en Drive esté COMPLETO.

    Este es el modo de falla característico de un montaje FUSE: Drive
    sincroniza la carpeta a medias, el explorador la muestra bien, pero falta
    un `.py` o llega con 0 bytes. Sin esta comprobación eso se manifiesta como
    un `ModuleNotFoundError` a los siete minutos de cómputo, con la corrida
    perdida. Aquí cuesta menos de un segundo y falla antes de leer un dato.

    Comprueba tres cosas distintas, porque una sincronización parcial se
    manifiesta de tres formas: subpaquete ausente, módulo ausente y archivo
    presente pero vacío (el placeholder que deja Drive mientras descarga).

    Args:
        raiz: carpeta del paquete (la que contiene `src/record_linkage`).

    Returns:
        Inventario medido: subpaquetes, módulos y archivos de caché.

    Raises:
        FileNotFoundError: si falta un subpaquete, faltan módulos o hay
            archivos truncados.
    """
    base = raiz / "src" / "record_linkage"
    faltantes = [
        p
        for p in SUBPAQUETES_ESPERADOS
        if not (base.joinpath(*p.split(".")) / "__init__.py").is_file()
    ]
    modulos = [p for p in base.rglob("*.py") if "__pycache__" not in p.parts]
    truncados = [p.relative_to(base) for p in modulos if p.stat().st_size == 0]
    caches = sum(1 for p in base.rglob("*") if p.name == "__pycache__")

    problemas: list[str] = []
    if faltantes:
        problemas.append(f"subpaquetes ausentes ({len(faltantes)}): {faltantes}")
    if len(modulos) < MODULOS_ESPERADOS:
        problemas.append(
            f"módulos: hay {len(modulos)} y se esperaban {MODULOS_ESPERADOS} "
            f"(faltan {MODULOS_ESPERADOS - len(modulos)})"
        )
    if truncados:
        problemas.append(
            f"archivos de 0 bytes ({len(truncados)}): "
            f"{[str(p) for p in truncados[:8]]}"
        )
    if problemas:
        raise FileNotFoundError(
            "Qué pasó: el árbol de rues-linker en Drive está incompleto.\n  · "
            + "\n  · ".join(problemas)
            + "\nPor qué importa: es la falla típica de FUSE. Sin este control, la "
            "corrida arranca y muere con ModuleNotFoundError a los minutos.\n"
            "Qué hacer: espere a que Drive termine de sincronizar (icono de nube en "
            "la app de Drive), o borre la carpeta en Drive y vuelva a subir el .zip "
            "descomprimido completo. Luego reinicie el entorno y repita esta celda."
        )

    print(
        f"🌳 árbol verificado: {len(SUBPAQUETES_ESPERADOS)} subpaquetes · "
        f"{len(modulos)} módulos · 0 truncados"
    )
    if caches:
        print(
            f"   ℹ️  hay {caches} carpetas __pycache__ en Drive: no estorban, pero son "
            "peso muerto de sincronización. Puede borrarlas."
        )
    return {"subpaquetes": len(SUBPAQUETES_ESPERADOS), "modulos": len(modulos), "caches": caches}


def importar_todo(paquete) -> dict[str, list[str]]:
    """Importa TODOS los módulos del paquete, ahora y no dentro de siete minutos.

    La verificación de archivos de `verificar_arbol` mira el disco; esta mira
    lo que Python realmente puede cargar. Un `.py` a medio sincronizar existe,
    pesa más de 0 bytes y aun así revienta al importarse con un SyntaxError o
    arrastra un import roto. La única forma de saberlo es importándolo.

    Los 6 módulos que dependen de extras no instalados (optuna, plotly,
    matplotlib, seaborn) se reportan como omitidos, no como falla: son
    opcionales por diseño y no participan en este flujo.

    Returns:
        Diccionario con las listas `omitidos` y `fallidos`.

    Raises:
        ImportError: si algún módulo del núcleo no se puede importar.
    """
    import pkgutil

    omitidos: list[str] = []
    fallidos: list[str] = []

    def _al_fallar(nombre: str) -> None:
        fallidos.append(f"{nombre} → error al recorrer el paquete")

    for info in pkgutil.walk_packages(paquete.__path__, paquete.__name__ + ".", _al_fallar):
        try:
            importlib.import_module(info.name)
        except ModuleNotFoundError as exc:
            raiz_faltante = (exc.name or "").split(".")[0]
            if raiz_faltante in EXTRAS_TOLERADOS:
                omitidos.append(f"{info.name} (requiere '{raiz_faltante}')")
            else:
                fallidos.append(f"{info.name} → falta el módulo '{exc.name}'")
        except Exception as exc:  # noqa: BLE001 — cualquier fallo aquí invalida el árbol
            fallidos.append(f"{info.name} → {type(exc).__name__}: {exc}")

    if fallidos:
        detalle = "\n  · ".join(fallidos[:12])
        raise ImportError(
            f"Qué pasó: {len(fallidos)} módulos de record_linkage no se pudieron "
            f"importar.\n  · {detalle}\n"
            "Por qué importa: el árbol de Drive está incompleto o corrupto. Si arranca "
            "así, la corrida muere a mitad de camino con la RAM ya comprometida.\n"
            "Qué hacer: espere a que Drive termine de sincronizar, o vuelva a subir el "
            ".zip descomprimido. Reinicie el entorno y repita esta celda."
        )

    print(
        f"📥 importación exhaustiva: todos los módulos del núcleo cargan "
        f"({len(omitidos)} omitidos por depender de un extra)"
    )
    for o in omitidos:
        print(f"   · omitido: {o}")
    return {"omitidos": omitidos, "fallidos": fallidos}


def asegurar_dependencias(raiz: Path) -> None:
    """Instala SOLO las dependencias que falten, leídas del pyproject del paquete.

    La lista no puede desincronizarse porque es la del propio paquete, no una
    copia escrita a mano en el notebook. Las versiones ya instaladas que quedan
    fuera del rango declarado se REPORTAN, no se tocan: degradar numpy o pandas
    en Colab obliga a reiniciar el entorno y es una decisión suya, no del script.
    """
    from packaging.requirements import Requirement

    declaradas = tomllib.loads((raiz / "pyproject.toml").read_text("utf-8"))
    faltantes: list[str] = []
    fuera_de_rango: list[str] = []
    for linea in declaradas["project"]["dependencies"]:
        req = Requirement(linea)
        try:
            instalada = version_de_distribucion(req.name)
        except PackageNotFoundError:
            faltantes.append(str(req))
            continue
        if req.specifier and not req.specifier.contains(instalada, prereleases=True):
            fuera_de_rango.append(f"{req.name} {instalada} · declarado {req.specifier}")

    if faltantes:
        nombres = ", ".join(Requirement(r).name for r in faltantes)
        print(f"📦 Faltan {len(faltantes)} dependencias · instalando: {nombres}")
        proc = subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", *faltantes],
            capture_output=True,
            text=True,
        )
        if proc.returncode != 0:
            raise RuntimeError(
                "Qué pasó: pip no pudo instalar las dependencias del motor.\n"
                "Por qué importa: sin ellas `import record_linkage` falla.\n"
                f"Qué hacer: revise el detalle y reintente.\n  {proc.stderr.strip()[-600:]}"
            )
        print("📦 Dependencias instaladas")
    else:
        print("📦 Dependencias: todas presentes")

    if fuera_de_rango:
        print("⚠️  Instaladas FUERA del rango declarado (no se modifican):")
        for x in fuera_de_rango:
            print(f"     · {x}")
        print("     Nota: el tope de numpy (<2.4) está documentado en pyproject como")
        print("     restricción del gate de tipos (mypy), no del runtime.")


def cargar_libreria(raiz: Path):
    """Pone `<raiz>/src` al frente de sys.path e importa el motor DESDE Drive."""
    src = str((raiz / "src").resolve())
    while src in sys.path:
        sys.path.remove(src)
    sys.path.insert(0, src)
    importlib.invalidate_caches()

    previo = sys.modules.get("record_linkage")
    if previo is not None and not str(Path(previo.__file__).resolve()).startswith(src):
        raise RuntimeError(
            f"Qué pasó: ya hay otro `record_linkage` cargado en memoria ({previo.__file__}).\n"
            "Por qué importa: seguiría corriendo ESE código, no el de Drive.\n"
            "Qué hacer: Entorno de ejecución → Reiniciar sesión, y repita esta celda."
        )

    import record_linkage

    origen = Path(record_linkage.__file__).resolve()
    if not str(origen).startswith(src):
        raise RuntimeError(
            f"Qué pasó: se importó `record_linkage` desde {origen}, no desde {src}.\n"
            "Por qué importa: no estaría auditando el código que subió a Drive.\n"
            "Qué hacer: `pip uninstall -y rues-linker`, reinicie la sesión y repita."
        )
    return record_linkage


# ── Ejecución ───────────────────────────────────────────────────────────────
DRIVE = montar_drive()
RAIZ_PAQUETE = raiz_del_paquete(RUTA_PAQUETE_DRIVE)
VERSION_EN_DRIVE = tomllib.loads(
    (RAIZ_PAQUETE / "pyproject.toml").read_text("utf-8")
)["project"]["version"]

if VERSION_EN_DRIVE != VERSION_OBJETIVO:
    raise RuntimeError(
        f"Qué pasó: el paquete en Drive declara {VERSION_EN_DRIVE} y este notebook está "
        f"escrito contra {VERSION_OBJETIVO}.\n"
        "Por qué importa: los valores por defecto medidos y los nombres de la API son de "
        "esa versión; mezclarlos invalida la trazabilidad de la corrida.\n"
        "Qué hacer: suba el paquete correcto a Drive, o cambie VERSION_OBJETIVO a conciencia."
    )

INVENTARIO = verificar_arbol(RAIZ_PAQUETE)
asegurar_dependencias(RAIZ_PAQUETE)
rl = cargar_libreria(RAIZ_PAQUETE)
IMPORTACION = importar_todo(rl)

print(f"\n🔌 motor  : rues-linker {VERSION_EN_DRIVE}  (versión declarada en el pyproject de Drive)")
print(f"📂 origen : {Path(rl.__file__).parent}")
if rl.__version__ != VERSION_EN_DRIVE:
    print(
        f"ℹ️  record_linkage.__version__ = '{rl.__version__}'. El paquete se importa desde\n"
        "    Drive SIN `pip install`, así que no hay metadatos de distribución que leer.\n"
        "    La versión válida es la del pyproject, verificada arriba."
    )

## 🧰 2 · Extras — las funciones auxiliares

Celda de **definición**: clases, funciones y constantes técnicas. Se corre una vez por
sesión y no contiene ni un solo valor que usted deba ajustar — esos viven todos en la
celda 3. La separación es deliberada: así una corrida se describe por completo con lo que
se ve en una sola pantalla de configuración.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
#  🧰 2 · EXTRAS — funciones auxiliares
#  Celda de DEFINICIÓN. Ejecutar UNA vez por sesión. Sin valores configurables.
# ══════════════════════════════════════════════════════════════════════════
from __future__ import annotations

import gc
import hashlib
import json
import time
from contextlib import contextmanager
from dataclasses import replace
from pathlib import Path

import pandas as pd

from record_linkage.exporters.smart import SmartExporter
from record_linkage.matching.normalizadores import PLACEHOLDERS
from record_linkage.flujo import (
    ConfigImportadores,
    PaisesSinClasificar,
    ResultadoImportadores,
    cobertura_paises,
    construir_entregables,
    deduplicar_importadores,
    exigir_cobertura_paises,
    grafias_aisladas,
    muestra_para_revision,
    preparar,
    recall_del_bloqueo,
    sensibilidad_umbral,
    verificar_invariantes,
)
from record_linkage.flujo import metricas as calcular_metricas
from record_linkage.flujo.importadores import ejecutar as ejecutar_emparejamiento
from record_linkage.paises import sugerir_alias_pais
from record_linkage.pipeline.result import PipelineResult
from record_linkage.utils.almacenamiento import copiar_si_existe

#: Extensiones que `leer_base` sabe abrir.
EXTENSIONES_SOPORTADAS: tuple[str, ...] = (".csv", ".txt", ".xlsx", ".xls", ".xlsm", ".parquet")

#: Tope de filas por hoja de Excel (el límite duro es 1.048.576; se deja margen).
LIMITE_EXCEL: int = 1_000_000

#: Valores que ocupan el lugar de una razón social sin serlo. No se eliminan
#: —sesgaría cualquier ranking—, se CUENTAN para que la decisión sea explícita.
#: Es la MISMA lista con la que el motor decide (`es_faltante` → grupo propio,
#: marcado `sin_nombre_utilizable`): hasta 0.22.3 había aquí una copia a mano y
#: la copia no coincidía con la del motor. Medido en snowflake_v2: "NO
#: DISPONIBLE" no estaba en ninguna de las dos y se convertía en "el importador
#: más grande" de 69 países, con el 21,7 % del FOB. Si necesita añadir uno, va
#: en `record_linkage.matching.normalizadores.PLACEHOLDERS`, con su prueba.
CENTINELAS_SIN_NOMBRE: frozenset[str] = PLACEHOLDERS


# ── Localización de la entrada ──────────────────────────────────────────────
def resolver_entrada(carpeta: str | Path, nombre: str = "") -> Path:
    """Devuelve la ruta del archivo de entrada en Drive.

    Con `nombre` vacío exige que la carpeta tenga exactamente un archivo con
    extensión soportada. Si hay varios NO elige por usted: enumera y se detiene.
    Elegir «el más reciente» haría que la corrida dependiera de un criterio que
    nadie declaró, y eso no se puede auditar después.

    Args:
        carpeta: carpeta de Drive con los insumos.
        nombre: nombre exacto del archivo; "" para detección automática.

    Returns:
        Ruta del archivo de entrada.

    Raises:
        FileNotFoundError: si la carpeta o el archivo no existen.
        ValueError: si hay más de un candidato y `nombre` está vacío.
    """
    carpeta = Path(carpeta)
    if not carpeta.is_dir():
        raise FileNotFoundError(
            f"Qué pasó: no existe la carpeta de datos {carpeta}.\n"
            "Por qué importa: sin archivo de entrada no hay nada que deduplicar.\n"
            "Qué hacer: verifique la ruta en Drive (tildes, espacios y mayúsculas cuentan)."
        )
    if nombre:
        ruta = carpeta / nombre
        if not ruta.is_file():
            visto = ", ".join(sorted(p.name for p in carpeta.iterdir())[:20]) or "(vacía)"
            raise FileNotFoundError(
                f"Qué pasó: no existe {ruta}.\n"
                f"Qué hacer: la carpeta contiene: {visto}. Corrija ARCHIVO_ENTRADA."
            )
        return ruta

    candidatos = sorted(
        p
        for p in carpeta.iterdir()
        if p.is_file()
        and p.suffix.lower() in EXTENSIONES_SOPORTADAS
        and not p.name.startswith((".", "~$"))
    )
    if not candidatos:
        raise FileNotFoundError(
            f"Qué pasó: {carpeta} no tiene ningún archivo {EXTENSIONES_SOPORTADAS}.\n"
            "Qué hacer: suba la base de importadores a esa carpeta de Drive."
        )
    if len(candidatos) > 1:
        listado = "\n".join(f"     · {p.name}" for p in candidatos)
        raise ValueError(
            f"Qué pasó: hay {len(candidatos)} archivos candidatos en {carpeta}.\n"
            "Por qué importa: elegir uno por usted haría que la corrida dependiera de un "
            "criterio que nadie declaró.\n"
            f"Qué hacer: ponga el nombre exacto en ARCHIVO_ENTRADA. Candidatos:\n{listado}"
        )
    return candidatos[0]


# ── Lectura y contrato ──────────────────────────────────────────────────────
def leer_base(
    ruta: str | Path,
    *,
    separador: str = ",",
    encoding: str = "utf-8-sig",
    hoja: int | str = 0,
) -> pd.DataFrame:
    """Lee CSV/TXT/XLSX/Parquet.

    El texto se lee como cadena (`dtype=str`) a propósito: pandas convertiría un
    NIT o un código de país numérico a float y perdería ceros a la izquierda. Las
    columnas de métricas se convierten aparte, y midiendo, en `convertir_metricas`.
    """
    p = Path(ruta)
    ext = p.suffix.lower()
    if ext in (".csv", ".txt"):
        return pd.read_csv(p, dtype=str, sep=separador, encoding=encoding)
    if ext in (".xlsx", ".xls", ".xlsm"):
        return pd.read_excel(p, sheet_name=hoja, dtype=str)
    if ext == ".parquet":
        return pd.read_parquet(p)
    raise ValueError(
        f"Qué pasó: extensión '{ext}' no soportada.\n"
        f"Qué hacer: use una de {EXTENSIONES_SOPORTADAS}."
    )


def verificar_contrato(df: pd.DataFrame, cfg: ConfigImportadores) -> None:
    """Falla ANTES de la corrida si falta alguna columna declarada en la celda 3.

    El motor haría la misma verificación, pero hacerla aquí permite mostrar de
    una vez las columnas reales del archivo para que el arreglo sea inmediato.
    """
    declaradas = [cfg.col_razon_social, cfg.col_pais, *cfg.cols_metricas]
    faltantes = [c for c in declaradas if c and c not in df.columns]
    if faltantes:
        raise KeyError(
            f"Qué pasó: faltan en el archivo las columnas {faltantes}.\n"
            "Por qué importa: son el contrato de entrada; sin ellas el motor no arranca.\n"
            "Qué hacer: corrija el perfil en la celda 3. El archivo trae:\n"
            f"  {list(df.columns)}"
        )
    print(f"✅ contrato OK · nombre={cfg.col_razon_social} · país={cfg.col_pais} · "
          f"métricas={list(cfg.cols_metricas)}")


def convertir_metricas(
    df: pd.DataFrame,
    columnas: tuple[str, ...],
    *,
    tolerancia: float = 0.0,
) -> pd.DataFrame:
    """Convierte las métricas a numérico MIDIENDO lo que se pierde.

    Por qué existe esta función: `flujo.importadores.preparar` hace
    `pd.to_numeric(..., errors="coerce").fillna(0)` (importadores.py, línea 345).
    Un valor que no parsea se vuelve **0 en silencio**. Si `USD_FOB_TOTAL` llega
    con separador de miles ("1,234,567.89") o coma decimal, el peso económico
    —que desempata el nombre final del grupo y ordena el GOLDEN— quedaría en cero
    sin un solo mensaje. Aquí la conversión es explícita, se cuenta y se detiene
    la corrida si supera la tolerancia.

    Args:
        df: base leída.
        columnas: columnas de métricas declaradas en la configuración.
        tolerancia: fracción de valores no vacíos que puede fallar sin detener
            la corrida. 0.0 = ninguno.

    Returns:
        Copia del DataFrame con las métricas ya numéricas.

    Raises:
        ValueError: si la fracción no convertible supera `tolerancia`.
    """
    out = df.copy()
    filas = []
    for col in columnas:
        serie = out[col]
        if pd.api.types.is_numeric_dtype(serie):
            convertida = serie.fillna(0)
            perdidos, no_vacios = 0, int(serie.notna().sum())
        else:
            texto = serie.astype("string").str.strip()
            no_vacio = texto.notna() & (texto != "")
            numerica = pd.to_numeric(texto, errors="coerce")
            fallidos = no_vacio & numerica.isna()
            perdidos, no_vacios = int(fallidos.sum()), int(no_vacio.sum())
            if perdidos:
                ejemplos = texto[fallidos].drop_duplicates().head(5).tolist()
                tasa = perdidos / max(no_vacios, 1)
                if tasa > tolerancia:
                    raise ValueError(
                        f"Qué pasó: {perdidos:,} de {no_vacios:,} valores de '{col}' "
                        f"({tasa:.2%}) no son numéricos.\n"
                        "Por qué importa: el motor los convierte a 0 en silencio; el peso "
                        "económico y el orden del GOLDEN quedarían mal sin ningún aviso.\n"
                        f"Qué hacer: revise el export (¿separador de miles?, ¿coma decimal?). "
                        f"Ejemplos: {ejemplos}"
                    )
            convertida = numerica.fillna(0)
        out[col] = convertida
        filas.append((col, no_vacios, perdidos, float(convertida.sum())))

    print("🔢 métricas convertidas a numérico:")
    for col, no_vacios, perdidos, total in filas:
        aviso = f"  ⚠️ {perdidos:,} no numéricos → 0" if perdidos else ""
        print(f"   · {col:<24} no vacíos {no_vacios:>12,}   suma {total:>22,.2f}{aviso}")
    print("   Concilie estas sumas contra Snowflake antes de seguir.")
    return out


def perfilar_entrada(df: pd.DataFrame, cfg: ConfigImportadores) -> dict[str, int]:
    """Reporte de composición de la entrada. Devuelve las cifras para la metadata.

    Mide lo que el `WHERE ... IS NOT NULL` de la consulta no filtra: cadenas
    vacías, centinelas del tipo '0' o 'N/A', países nulos y parejas repetidas.
    Una pareja repetida significa que el GROUP BY del origen no coincide con el
    contrato declarado aquí, y eso invalida las métricas agregadas.
    """
    nombre = df[cfg.col_razon_social].astype("string").str.strip().str.upper()
    pais = df[cfg.col_pais].astype("string").str.strip().str.upper()
    sin_nombre = int(nombre.isna().sum() + nombre.fillna("").isin(CENTINELAS_SIN_NOMBRE).sum())
    sin_pais = int(pais.isna().sum() + (pais.fillna("") == "").sum())
    repetidas = int(df.duplicated(subset=[cfg.col_razon_social, cfg.col_pais]).sum())

    stats = {
        "filas": len(df),
        "columnas": len(df.columns),
        "paises_distintos": int(pais.nunique(dropna=True)),
        "nombres_distintos": int(nombre.nunique(dropna=True)),
        "sin_nombre_utilizable": sin_nombre,
        "sin_pais": sin_pais,
        "parejas_repetidas": repetidas,
    }
    print(
        f"\n📊 Composición de la entrada\n"
        f"   • Filas                     : {stats['filas']:>12,}\n"
        f"   • Razones sociales distintas: {stats['nombres_distintos']:>12,}\n"
        f"   • Países distintos          : {stats['paises_distintos']:>12,}\n"
        f"   • Sin nombre utilizable     : {stats['sin_nombre_utilizable']:>12,}"
        f"  ({stats['sin_nombre_utilizable'] / max(len(df), 1):.2%})\n"
        f"   • Sin país                  : {stats['sin_pais']:>12,}\n"
        f"   • Parejas (nombre,país) rep.: {stats['parejas_repetidas']:>12,}"
    )
    if repetidas:
        print(
            "   ⚠️  Hay parejas repetidas: el GROUP BY del origen NO coincide con el\n"
            "       contrato declarado. Las métricas quedarían partidas entre filas."
        )
    if sin_nombre:
        print(
            "   ⚠️  Los centinelas ('0', 'N/A', vacío) quedan cada uno en su propio grupo,\n"
            "       marcados `sin_nombre_utilizable`. No se eliminan: sesgaría el ranking."
        )
    return stats


# ── Cronometraje y observabilidad ───────────────────────────────────────────
def reportar_cobertura_paises(df: pd.DataFrame, cfg: ConfigImportadores) -> pd.DataFrame:
    """Grafías de país que el catálogo no reconoce, sobre la base COMPLETA.

    Va antes del smoke test y no dentro de él, por dos razones: el smoke test
    corre sobre una muestra —vería 3 filas donde hay 300— y una grafía sin
    clasificar no es un fallo del pipeline sino una decisión suya (catalogar,
    declarar como no-país, o aislar). La decisión la aplica la librería con
    `exigir_cobertura_paises`, que es exactamente lo que `preparar()` hace por
    dentro: una sola lógica, dos puntos de llamada.

    Returns:
        La tabla (`valor`, `n_filas`, `sugerencia`, `similitud`); vacía si la
        cobertura es total.

    Raises:
        PaisesSinClasificar: si hay grafías y `cfg.paises_sin_clasificar` es
            `"detener"`. El mensaje trae la lista y los tres remedios.
    """
    tabla = cobertura_paises(df, cfg)
    if tabla.empty:
        print(f"🌍 catálogo de países: cobertura total ({df[cfg.col_pais].nunique():,} grafías)")
        return tabla
    filas = int(tabla.n_filas.sum())
    print(
        f"🌍 catálogo de países: {len(tabla)} grafía(s) sin clasificar · "
        f"{filas:,} filas ({filas / len(df):.2%})  ·  modo: {cfg.paises_sin_clasificar}"
    )
    print(tabla.head(25).to_string(index=False))
    if len(tabla) > 25:
        print(f"   … y {len(tabla) - 25} grafías más (todas en RESULTADO.sugerencias_pais)")
    exigir_cobertura_paises(tabla, cfg)  # se detiene si hay alguna NO declarada
    declaradas = tabla[tabla.valor.isin(grafias_aisladas(cfg))]
    if len(declaradas):
        print(
            f"   ✅ {len(declaradas)} declarada(s) en NO_SON_PAISES (celda 4) → van aparte, "
            f"con su propio PAIS_FINAL: {', '.join(declaradas.valor)}"
        )
    if cfg.paises_sin_clasificar == "aislar":
        print(
            "   ⚠️  modo 'aislar': TODO lo no catalogado va aparte, declarado o no. Para\n"
            "      producción use 'detener' y declare en la celda 4 lo que no sea país."
        )
    return tabla


def formato_duracion(segundos: float) -> str:
    """Duración legible: '12,3 s', '3 min 05 s', '1 h 04 min'."""
    if segundos < 60:
        return f"{segundos:,.1f} s"
    minutos, seg = divmod(int(round(segundos)), 60)
    if minutos < 60:
        return f"{minutos} min {seg:02d} s"
    horas, minutos = divmod(minutos, 60)
    return f"{horas} h {minutos:02d} min"


class Cronometro:
    """Mide cada fase y reparte el total.

    Por qué un gestor de contexto y no un par de `time.time()`: la marca queda
    registrada aunque la fase lance una excepción. En una corrida de minutos,
    saber en QUÉ fase murió vale más que el total que nunca llegó.

    Volver a correr una celda REEMPLAZA la marca de esa fase en vez de
    duplicarla: en Colab una celda se ejecuta varias veces y el reparto debe
    seguir describiendo la última corrida.
    """

    def __init__(self, silencioso: bool = False) -> None:
        self.marcas: dict[str, float] = {}
        self.silencioso = silencioso

    @contextmanager
    def fase(self, nombre: str):
        """Cronometra el bloque y lo anuncia al entrar y al salir."""
        if not self.silencioso:
            print(f"\n▶️  {nombre}", flush=True)
        t0 = time.perf_counter()
        try:
            yield self
        except BaseException:
            self.marcas[nombre] = time.perf_counter() - t0
            if not self.silencioso:
                print(f"   ❌ falló tras {formato_duracion(self.marcas[nombre])}", flush=True)
            raise
        self.marcas[nombre] = time.perf_counter() - t0
        if not self.silencioso:
            print(f"   ⏱️  {formato_duracion(self.marcas[nombre])}", flush=True)

    @property
    def total(self) -> float:
        """Suma de todas las fases registradas, en segundos."""
        return float(sum(self.marcas.values()))

    def resumen(self) -> pd.DataFrame:
        """Tabla fase / segundos / % del total, con la fila TOTAL al final."""
        total = self.total or 1.0
        filas = [(n, round(s, 2), round(100 * s / total, 1)) for n, s in self.marcas.items()]
        filas.append(("TOTAL", round(self.total, 2), 100.0))
        return pd.DataFrame(filas, columns=["fase", "segundos", "pct_del_total"])

    def imprimir_resumen(self, titulo: str = "⏱️  TIEMPOS POR FASE") -> None:
        """Reparto del tiempo con barras, para ver de un vistazo dónde se fue."""
        if not self.marcas:
            print("(sin fases cronometradas)")
            return
        ancho = max(len(n) for n in self.marcas)
        total = self.total or 1.0
        print(f"\n{titulo}")
        print("   " + "─" * (ancho + 46))
        for nombre, seg in self.marcas.items():
            barra = "█" * max(1, round(24 * seg / total))
            print(f"   {nombre:<{ancho}}  {formato_duracion(seg):>12}  "
                  f"{100 * seg / total:>5.1f}%  {barra}")
        print("   " + "─" * (ancho + 46))
        print(f"   {'TOTAL':<{ancho}}  {formato_duracion(self.total):>12}")


def panorama_particiones(prep: pd.DataFrame, top: int = 10) -> None:
    """Muestra la FORMA del trabajo antes de la fase larga.

    El costo del emparejamiento no depende de las filas sino de los nombres
    únicos por país: dentro de cada partición el bloqueo compara nombre contra
    nombre. Ver de antemano las particiones grandes explica dónde se va el
    tiempo y permite abortar a tiempo si una está desbocada.
    """
    rep = prep.loc[prep["NOMBRE_NORM"] != ""].drop_duplicates(["PAIS_ISO3", "NOMBRE_NORM"])
    tam = rep.groupby("PAIS_ISO3").size().sort_values(ascending=False)
    pares_fb = int((tam * (tam - 1) // 2).sum())
    print("\n🗺️  Panorama del emparejamiento")
    print(f"   • Particiones (países)        : {len(tam):>14,}")
    print(f"   • Nombres únicos a comparar   : {int(tam.sum()):>14,}")
    print(f"   • Pares si fuera fuerza bruta : {pares_fb:>14,}   (el LSH reduce 94–97 %)")
    print(f"   • Las {min(top, len(tam))} particiones más grandes:")
    for iso, n in tam.head(top).items():
        print(f"       {iso}  {n:>10,} nombres")
    print("   ⏳ El avance se imprime cada 25 particiones, sobrescribiendo la misma línea.")


def ejecutar_por_fases(
    df: pd.DataFrame,
    cfg: ConfigImportadores,
    crono: Cronometro | None = None,
    *,
    panorama: bool = True,
) -> ResultadoImportadores:
    """Corre el pipeline etapa por etapa, cronometrando cada una.

    Por qué no se llama a la fachada: `deduplicar_importadores` encadena las
    ocho etapas en una sola llamada, así que desde afuera solo se puede medir
    el total. Esta función ejecuta exactamente las mismas etapas, en el mismo
    orden y con los mismos argumentos que la fachada `deduplicar_importadores`
    de rues-linker 0.22.4— y devuelve el mismo `ResultadoImportadores`.

    El riesgo de reproducir una fachada es que se desincronice de la librería.
    Está acotado por dos cosas: la celda 1 detiene el notebook si el paquete no
    es exactamente 0.22.4, y el smoke test compara esta ruta contra la fachada
    sobre una muestra y avisa si difieren. Si algún día divergen, ponga
    `USAR_FASES = False` en la celda 3 y se vuelve a la fachada.

    Args:
        df: base cruda, ya con las métricas convertidas a numérico.
        cfg: configuración del motor.
        crono: cronómetro donde registrar las fases; se crea uno si falta.
        panorama: imprimir el tamaño de las particiones antes de emparejar.

    Returns:
        El mismo `ResultadoImportadores` que devolvería la fachada.
    """
    crono = crono if crono is not None else Cronometro()

    with crono.fase("1 · Preparación (saneamiento, país canónico, normalización)"):
        prep = preparar(df, cfg)

    with crono.fase("2 · Sugerencias para países sin clasificar"):
        sugerencias = sugerir_alias_pais(
            prep.loc[prep.PAIS_METODO == "sin_clasificar", "PAIS_NORMALIZADO"],
            catalogo=cfg.catalogo_paises,
        )

    if panorama:
        panorama_particiones(prep)

    with crono.fase("3 · Emparejamiento: bloqueo LSH + scoring + clustering"):
        corrida = ejecutar_emparejamiento(prep, cfg)

    with crono.fase("4 · Construcción de entregables (correlativa, golden, países)"):
        tablas = construir_entregables(prep, corrida, cfg)

    with crono.fase("5 · Muestra estratificada para revisión manual"):
        muestra = muestra_para_revision(tablas, cfg)

    with crono.fase("6 · Sensibilidad al umbral"):
        sensibilidad = sensibilidad_umbral(corrida, cfg)

    with crono.fase("7 · Recall del bloqueo (fuerza bruta)"):
        recall = recall_del_bloqueo(corrida, cfg)

    with crono.fase("8 · Métricas e invariantes"):
        tabla_metricas = calcular_metricas(prep, tablas, corrida, cfg)
        tabla_invariantes = verificar_invariantes(prep, tablas, cfg)

    extra: dict[str, pd.DataFrame] = {
        "SENSIBILIDAD": sensibilidad,
        "RECALL_BLOQUEO": recall,
        "PARAMETROS": pd.DataFrame(
            [(k, str(v)) for k, v in sorted(vars(cfg).items())],
            columns=["parametro", "valor"],
        ),
    }
    if len(sugerencias):
        extra["SUGERENCIAS_PAIS"] = sugerencias

    return ResultadoImportadores(
        correlativa=tablas["CORRELATIVA"],
        golden=tablas["GOLDEN"],
        paises=tablas["PAISES"],
        revision=tablas["REVISION"],
        muestra=muestra,
        metricas=tabla_metricas,
        invariantes=tabla_invariantes,
        decisiones=corrida["decisiones"],
        comparador=corrida["comparador"],
        preparada=prep,
        representantes=corrida["representantes"],
        sugerencias_pais=sugerencias,
        segundos=corrida["segundos"],
        n_candidatos=corrida["n_candidatos"],
        n_cortes_cohesion=corrida["n_cortes"],
        extra=extra,
    )


def smoke_test(df: pd.DataFrame, cfg: ConfigImportadores, n: int = 5_000) -> bool:
    """Corre el pipeline completo sobre una muestra antes de pagar la corrida real.

    Hace dos cosas en segundos que de otro modo se descubrirían en minutos:
    valida rutas, tipos, contrato e invariantes; y comprueba que la ruta
    cronometrada por fases produce EXACTAMENTE el mismo resultado que la
    fachada `deduplicar_importadores`. Esa segunda comprobación es la que
    impide que el cronómetro se convierta en una copia divergente del motor.

    Se desactiva la auditoría de bloqueo (O(k²)) porque sobre una muestra no
    mide nada útil.

    Returns:
        True si el pipeline corrió y todas las invariantes pasaron.
    """
    muestra = df.sample(n=min(n, len(df)), random_state=cfg.semilla)
    cfg_prueba = replace(cfg, paises_auditoria_bloqueo=(), verboso=False)
    try:
        por_fases = ejecutar_por_fases(
            muestra, cfg_prueba, Cronometro(silencioso=True), panorama=False
        )
        fachada = deduplicar_importadores(muestra, cfg_prueba)
    except PaisesSinClasificar as exc:
        # No es un fallo del pipeline: es la decisión de la celda 3 aplicada a
        # una muestra. La celda 6 ya la comprobó sobre la base completa; si se
        # llega aquí es porque alguien saltó ese paso.
        print(f"❌ Smoke test FALLÓ por países sin clasificar:\n{exc}")
        return False
    except Exception as exc:  # noqa: BLE001 — el smoke test debe reportar, no propagar
        print(f"❌ Smoke test FALLÓ: {type(exc).__name__}: {exc}")
        return False

    if not por_fases.todo_ok:
        print("❌ Smoke test FALLÓ: alguna invariante no se cumple sobre la muestra.")
        print(por_fases.invariantes.to_string(index=False))
        return False

    iguales = por_fases.correlativa.equals(fachada.correlativa) and por_fases.golden.equals(
        fachada.golden
    )
    print(
        f"✅ Smoke test OK · {len(muestra):,} filas → {len(por_fases.golden):,} importadores "
        f"en {por_fases.segundos:.1f} s"
    )
    if iguales:
        print("   ✅ la ruta por fases produce el mismo resultado que la fachada")
    else:
        print(
            "   ⚠️  la ruta por fases DIFIERE de la fachada sobre esta muestra.\n"
            "       Ponga USAR_FASES = False en la celda 3 (se pierde el detalle por\n"
            "       fase, se conserva el total) y avise: el motor cambió de forma."
        )
    return True


# ── Estado y trazabilidad ───────────────────────────────────────────────────
def limpiar_estado(*nombres: str) -> None:
    """Libera variables pesadas del espacio global antes de recalcular.

    En Colab las variables sobreviven entre ejecuciones de celda: volver a correr
    la celda 7 sin soltar el RESULTADO anterior duplica el pico de RAM.
    """
    entorno = globals()
    borradas = [n for n in nombres if n in entorno]
    for n in borradas:
        del entorno[n]
    gc.collect()
    if borradas:
        print(f"🧽 estado previo liberado: {', '.join(borradas)}")


def sha256_archivo(ruta: str | Path, bloque: int = 1 << 20) -> str:
    """SHA-256 del archivo, leído por bloques (no carga el archivo en RAM)."""
    h = hashlib.sha256()
    with open(ruta, "rb") as f:
        for trozo in iter(lambda: f.read(bloque), b""):
            h.update(trozo)
    return h.hexdigest()


# ── Exportación ─────────────────────────────────────────────────────────────
def exportar(
    tablas: dict[str, pd.DataFrame],
    dir_trabajo: str | Path,
    dir_drive: str | Path,
    nombre_base: str,
    metadata: dict | None = None,
) -> list[Path]:
    """Parquet primero (checkpoint), luego XLSX, metadata y copia a Drive.

    Orden deliberado: el Parquet es barato, no pierde tipos y queda escrito antes
    de que el XLSX —que es la parte lenta y la que puede reventar por tamaño—
    tenga oportunidad de fallar. Si la sesión muere durante el Excel, el
    resultado de la corrida ya está en disco.

    `PipelineResult`/`SmartExporter` neutralizan el texto que empieza por `=`,
    `+`, `-`, `@`, tabulador o salto de línea antes de escribir: una razón social
    puede empezar por `+` —en la base de referencia una lo hace— y Excel la
    interpretaría como fórmula. Y quitan los caracteres de control que openpyxl
    rechaza (0.22.4): diez razones sociales de la base real traen `\x1a`, el
    mojibake de "Ñ", y el XLSX no se escribía. El Parquet conserva el texto tal cual.

    Args:
        tablas: las tablas del resultado (`RESULTADO.tablas()`).
        dir_trabajo: directorio local de escritura intensiva.
        dir_drive: destino final en Drive.
        nombre_base: prefijo común de los archivos de esta corrida.
        metadata: diccionario de trazabilidad; se escribe como `*_METADATA.json`.

    Returns:
        Rutas efectivamente disponibles en Drive.
    """
    dir_trabajo, dir_drive = Path(dir_trabajo), Path(dir_drive)
    dir_trabajo.mkdir(parents=True, exist_ok=True)
    dir_drive.mkdir(parents=True, exist_ok=True)
    locales: list[Path] = []

    # 1. Parquet — checkpoint del resultado. CORRELATIVA es obligatoria.
    for hoja, tabla in tablas.items():
        destino = dir_trabajo / f"{nombre_base}__{hoja}.parquet"
        try:
            tabla.to_parquet(destino, index=False)
            locales.append(destino)
        except Exception as exc:  # noqa: BLE001
            if hoja == "CORRELATIVA":
                raise
            print(f"  ⚠️  {hoja}: no se pudo guardar en Parquet ({exc}); sigue en el XLSX.")
    print(f"  💾 checkpoint: {len(locales)} tablas en Parquet")

    # 2. XLSX multi-hoja si todo cabe; si no, un CSV.gz por tabla.
    vacias = [k for k, t in tablas.items() if not len(t)]
    if all(len(t) <= LIMITE_EXCEL for t in tablas.values()):
        resultado = PipelineResult(work_dir=dir_trabajo, extra=dict(tablas))
        locales.append(
            Path(resultado.to_excel(dir_trabajo / f"{nombre_base}.xlsx", include=tuple(tablas)))
        )
    else:
        grandes = {k: len(t) for k, t in tablas.items() if len(t) > LIMITE_EXCEL}
        print(f"  ℹ️  {grandes} superan el límite de Excel → CSV.gz por tabla")
        exportador = SmartExporter({"output_directory": str(dir_trabajo)})
        for hoja, tabla in tablas.items():
            if len(tabla):
                locales.append(
                    Path(exportador.export(tabla, f"{nombre_base}__{hoja}", format="csv.gz"))
                )

    # 3. Metadata de la corrida.
    if metadata is not None:
        ruta_meta = dir_trabajo / f"{nombre_base}__METADATA.json"
        ruta_meta.write_text(
            json.dumps(metadata, indent=2, ensure_ascii=False, default=str), encoding="utf-8"
        )
        locales.append(ruta_meta)

    # 4. Local → Drive. Si la copia falla, el archivo local sigue siendo válido.
    en_drive: list[Path] = []
    for ruta in locales:
        destino = dir_drive / ruta.name
        if ruta.resolve() == destino.resolve() or copiar_si_existe(ruta, destino):
            en_drive.append(destino)
        else:
            print(f"  ⚠️  no se pudo copiar a Drive: {ruta.name}  ·  queda en {ruta.parent}")
    for r in en_drive:
        print(f"  ✅ {r.name}")
    if vacias:
        print(f"  ℹ️  tablas vacías (sin hoja en el XLSX): {', '.join(vacias)}")
    return en_drive


print(
    "🧰 Extras cargados\n"
    "   entrada    : resolver_entrada · leer_base · verificar_contrato · convertir_metricas\n"
    "   diagnóstico: perfilar_entrada · reportar_cobertura_paises · panorama_particiones · smoke_test\n"
    "   tiempos    : Cronometro · formato_duracion\n"
    "   pipeline   : ejecutar_por_fases\n"
    "   salida     : limpiar_estado · sha256_archivo · exportar"
)

## 🎛️ 3 · Configuración — **la única celda que usted edita**

Tres bloques: **rutas**, **perfil de columnas** y **parámetros del motor**.

El perfil de columnas es un diccionario con nombre. Para una vista distinta de Snowflake
—o para volver a la base anterior— añada una entrada y cambie `PERFIL`: una línea, no
cuatro ediciones dispersas. Los nombres declarados aquí son los que usan la verificación
del contrato, la conversión de métricas y el perfilado; no hay una segunda copia en
ninguna parte.

Los valores del motor salen de una corrida medida sobre **211.949 destinatarios reales**,
no de un supuesto. **Regla de calibración:** cambie **un** parámetro a la vez y vuelva a
contar la muestra de la celda 8. Dos cambios simultáneos no se pueden atribuir.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
#  🎛️ 3 · CONFIGURACIÓN — LA CELDA QUE USTED EDITA
# ══════════════════════════════════════════════════════════════════════════

# ── 3.1 · Rutas ─────────────────────────────────────────────────────────────
CARPETA_ENTRADA = "/content/drive/MyDrive/ProColombia/0A. Datos/Importadores"  # @param {type:"string"}
CARPETA_SALIDA = "/content/drive/MyDrive/ProColombia/1B. Resultados/Rues-Linker/Importadores"  # @param {type:"string"}
ARCHIVO_ENTRADA = ""  # @param {type:"string"}  ·  "" = detectar el único archivo de la carpeta

#  Formato del archivo (solo aplica lo que corresponda a su extensión).
SEPARADOR_CSV = ","  # @param [",", ";", "|", "\t"]
ENCODING_CSV = "utf-8-sig"  # @param {type:"string"}
HOJA_EXCEL = 0  # @param {type:"raw"}

# ── 3.2 · Perfil de columnas (el contrato de entrada) ───────────────────────
#  Un perfil = los cuatro nombres de columna de una fuente. Para otra vista,
#  añada una entrada aquí y cambie PERFIL. No hay otra copia de estos nombres.
PERFILES_ENTRADA: dict[str, dict[str, object]] = {
    #  GOLD.VW_EXPORTACIONES_COLOMBIANAS_LECTURA_V2, FECHA 201601–202607,
    #  agrupada por (RAZON_SOCIAL_IMPORTADOR, PAIS_ESTANDAR).
    "snowflake_v2": {
        "col_razon_social": "RAZON_SOCIAL_IMPORTADOR",
        "col_pais": "PAIS_ESTANDAR",
        "cols_metricas": ("USD_FOB_TOTAL", "NUM_REGISTROS"),
        "col_peso_economico": "USD_FOB_TOTAL",
    },
    #  Base anterior de destinatarios de exportación (DIAN).
    "destinatarios_dian": {
        "col_razon_social": "RAZON_SOCIAL_DESTINATARIO",
        "col_pais": "PAIS_DESTINO_FINAL",
        "cols_metricas": ("NUM_REGISTROS_EXPORTACION", "VALOR_FOB_USD_TOTAL"),
        "col_peso_economico": "VALOR_FOB_USD_TOTAL",
    },
}
PERFIL = "snowflake_v2"  # @param ["snowflake_v2", "destinatarios_dian"]

#  Fracción de valores no vacíos de una métrica que puede fallar la conversión
#  a numérico sin detener la corrida. 0.0 = ninguno (recomendado).
TOLERANCIA_NO_NUMERICOS = 0.0  # @param {type:"number"}

#  Qué hacer con las grafías de país que el catálogo NO reconoce (celda 6 las
#  lista sobre la base completa, antes del smoke test):
#    "detener"  se detiene y le muestra cada grafía con sus filas y el país más
#               parecido. Usted decide: alias en la celda 4, patrón de no-país,
#               o pasar a "aislar". Es el defecto porque el país es el bloqueo
#               duro y una grafía sin catalogar fusiona lo que no debe (medido).
#    "aislar"   cada grafía conserva su propio PAIS_FINAL y no se fusiona con
#               nada; la corrida sigue. Para NO DEFINIDO, VARIOS y similares.
PAISES_SIN_CLASIFICAR = "detener"  # @param ["detener", "aislar"]

#  Smoke test sobre una muestra antes de la corrida completa.
SMOKE_TEST = True  # @param {type:"boolean"}
FILAS_SMOKE_TEST = 5_000  # @param {type:"integer"}

#  Sello de fecha y hora en los nombres de salida: una corrida nueva no pisa
#  la evidencia de la anterior.
VERSIONAR_SALIDA = True  # @param {type:"boolean"}

#  Ejecutar el pipeline etapa por etapa para cronometrar cada fase. False usa
#  la fachada `deduplicar_importadores`: mismo resultado, pero solo el total.
USAR_FASES = True  # @param {type:"boolean"}

#  Avance del motor en pantalla: filas leídas, cobertura de países, vocabulario,
#  contador de particiones y fusiones. Es `ConfigImportadores.verboso`.
MOSTRAR_AVANCE = True  # @param {type:"boolean"}

if PERFIL not in PERFILES_ENTRADA:
    raise KeyError(
        f"Qué pasó: el perfil '{PERFIL}' no está declarado.\n"
        f"Qué hacer: use uno de {list(PERFILES_ENTRADA)} o añádalo a PERFILES_ENTRADA."
    )
_COLUMNAS = PERFILES_ENTRADA[PERFIL]

# ── 3.3 · Parámetros del motor ──────────────────────────────────────────────
CFG = ConfigImportadores(
    #  Contrato de entrada: viene del perfil elegido arriba.
    col_razon_social=_COLUMNAS["col_razon_social"],
    col_pais=_COLUMNAS["col_pais"],
    cols_metricas=_COLUMNAS["cols_metricas"],
    col_peso_economico=_COLUMNAS["col_peso_economico"],
    #
    #  Grafías de país fuera del catálogo (ver PAISES_SIN_CLASIFICAR arriba).
    paises_sin_clasificar=PAISES_SIN_CLASIFICAR,
    #
    #  LA perilla. Medido sobre 160 asignaciones revisadas a mano:
    #    0.84 → precisión ponderada 0,966   (defecto)
    #    0.88 → más precisión, pero pierde ~7.000 empalmes correctos
    #    0.80 → el tramo añadido cae a ~0,75 de precisión
    umbral_nombre=0.84,
    #
    #  Cuánto pesa la evidencia de TOKENS frente a la de CADENA. Subirlo castiga
    #  más los prefijos genéricos; bajarlo perdona más erratas.
    alfa_idf=0.25,
    #
    #  ¿La geografía es ruido? Con True, "ECOLAB" y "ECOLAB CHILE" se unen — y
    #  también "BARRY CALLEBAUT USA" con "BARRY CALLEBAUT CANADA", que son dos
    #  sociedades del mismo grupo. Es el modo de error dominante de la banda
    #  0,92–0,96 (4 de 40 revisadas). False = resolver a nivel de PERSONA
    #  JURÍDICA y no de grupo comercial.
    geografia_es_ruido=True,
    #
    #  Bloqueo. Medido por fuerza bruta sobre particiones completas (PC = recall):
    #    128 permutaciones @ 0.25 → PC 0,99–1,00   (defecto)
    #     64 permutaciones @ 0.30 → PC 0,97–0,99   (más rápido)
    #     64 permutaciones @ 0.35 → PC 0,67–0,81   ← NO usar: pierde 1 de cada 4
    lsh_permutaciones=128,
    lsh_umbral=0.25,
    #
    #  Control de calidad. La auditoría del bloqueo es O(k²) por partición:
    #  elija particiones medianas (1.000–4.000 nombres) o déjela vacía.
    paises_auditoria_bloqueo=("GTM", "NLD", "DEU"),
    muestra_revision_por_banda=40,
    semilla=42,
    #
    #  Avance en pantalla durante la corrida (ver MOSTRAR_AVANCE arriba).
    verboso=MOSTRAR_AVANCE,
)

print(f"🧾 perfil de entrada : {PERFIL}")
print(f"   nombre  : {CFG.col_razon_social}")
print(f"   país    : {CFG.col_pais}")
print(f"   métricas: {list(CFG.cols_metricas)}   (peso económico: {CFG.col_peso_economico})")
print(f"\n🎯 umbral de nombre {CFG.umbral_nombre}  →  umbral de score {CFG.umbral_score:.4f}")
print(f"🧱 bloqueo LSH: {CFG.lsh_permutaciones} permutaciones @ {CFG.lsh_umbral}")
print(f"🌐 geografía como ruido: {CFG.geografia_es_ruido}")
print(f"🌍 países sin clasificar: {CFG.paises_sin_clasificar}")
print(f"⏱️  cronómetro por fases: {USAR_FASES}   ·   avance en pantalla: {CFG.verboso}")

## 🌍🧹 4 y 5 · Los dos diccionarios editables *(opcionales)*

El catálogo de países y las listas de limpieza son **datos**, no código. Viven en
`record_linkage.paises` y `record_linkage.matching.genericos`, con pruebas; estas dos
celdas solo los **amplían** para su base.

Si no le falta ninguna grafía y no ve grupos absorbidos por una palabra corriente,
**sáltelas**: los valores de la librería ya funcionan.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
#  🌍 4 · CATÁLOGO DE PAÍSES   (alias que falten, y lo que NO es un país)
# ══════════════════════════════════════════════════════════════════════════
#
#  La librería trae 214 países y 378 grafías en `record_linkage.paises`. Es un
#  DATO versionado, no una inferencia: hay pares que son el mismo país y no se
#  parecen (TURQUIA/TÜRKIYE, CHEQUIA/REPÚBLICA CHECA, YIBUTI/DJIBOUTI) y pares
#  que se parecen mucho y son países distintos (GUINEA/GUINEA-BISSAU,
#  CONGO/REPÚBLICA DEMOCRÁTICA DEL CONGO). Ninguna similitud de cadenas acierta
#  en los dos casos a la vez.
#
#  `PAIS_ESTANDAR` viene estandarizado desde Snowflake, pero estandarizado no es
#  catalogado: la primera corrida trajo 19 grafías fuera del catálogo. 18 eran
#  países que faltaban y ya están en la librería (0.22.4). La celda 6 le dirá
#  si aparece alguna nueva, ANTES de emparejar.

from record_linkage.paises import CATALOGO_PAISES

#  Formato: (ISO 3166-1 alfa-3, NOMBRE CANÓNICO, (alias adicionales, ...)).
#  Los alias se comparan sin tildes, en mayúsculas y sin puntuación.
GRAFIAS_ADICIONALES: tuple[tuple[str, str, tuple[str, ...]], ...] = (
    # ("USA", "ESTADOS UNIDOS", ("EE.UU.", "ESTADOS UNIDOS DE NORTEAMERICA")),
    # ("CHN", "CHINA", ("REPUBLICA POPULAR DE CHINA",)),
)

CFG.catalogo_paises = CATALOGO_PAISES + GRAFIAS_ADICIONALES

#  Grafías que NO son un país y no tiene sentido catalogar. Van aparte, cada
#  una con su propio PAIS_FINAL ("SIN CLASIFICAR: OTROS"): se deduplican por
#  nombre entre sí y NUNCA se fusionan con un país real. Cualquier OTRA grafía
#  fuera del catálogo sigue deteniendo la celda 6 — declarar OTROS no apaga la
#  guardia. En snowflake_v2, OTROS son 444 filas (0,12 %).
NO_SON_PAISES: tuple[str, ...] = ("OTROS",)
CFG.paises_aislar = NO_SON_PAISES

print(
    f"🌍 catálogo: {len(CFG.catalogo_paises)} países · "
    f"{sum(1 + len(a) for _, _, a in CFG.catalogo_paises)} grafías declaradas "
    f"(+{len(GRAFIAS_ADICIONALES)} suyas)\n"
    f"🚫 no son países (van aparte): {list(NO_SON_PAISES) or 'ninguna'}"
)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
#  🧹 5 · LISTAS DE LIMPIEZA   (la segunda palanca de calidad; opcional)
# ══════════════════════════════════════════════════════════════════════════
#
#  Tres listas, tres papeles distintos. No las mezcle:
#
#   1. SUFIJOS     se BORRAN del nombre (formas legales: LLC, SAS, GMBH...).
#   2. GENÉRICOS   NO se borran: se les baja el peso a casi cero.
#   3. SECTOR      se ponderan a la baja, pero NO cuentan como ruido al
#                  comparar: son lo único que separa "GREENWAY FARMS" de
#                  "GREENWAY COFFEE", dos empresas distintas.
#
#  Por qué DECLARADAS y no inferidas del corpus: el IDF aprendido no distingue
#  una marca de un genérico. Medido sobre 105.705 razones sociales,
#  IMPORTADORA (737 apariciones) pesa 5,96 de IDF y ZELECTA (490) pesa 6,37 —
#  la marca pesa MENOS que el genérico, porque la frecuencia de una marca crece
#  con el número de variantes de la MISMA empresa. El corpus la castiga por el
#  motivo equivocado.

from record_linkage.matching.genericos import (
    GENERICOS_ESTRUCTURALES,
    GENERICOS_GEOGRAFIA,
    GENERICOS_SECTOR,
    SUFIJOS_INTERNACIONALES,
)

#  Términos propios de SU dominio. Añada aquí lo que vea absorbiendo grupos en
#  la revisión de la celda 8.
GENERICOS_PROPIOS: frozenset[str] = frozenset({
    # "MAQUILA", "ZOFRI", "TRANSITARIO",
})

#  Formas legales que su base traiga y la librería no cubra.
SUFIJOS_PROPIOS: frozenset[str] = frozenset({
    # "ANONIM SIRKETI", "KABUSHIKI KAISHA",
})

CFG.genericos_extra = GENERICOS_PROPIOS
CFG.sufijos_extra = SUFIJOS_INTERNACIONALES | SUFIJOS_PROPIOS

print(
    f"🧹 sufijos: {len(CFG.sufijos_extra)} internacionales (+ los ES/EN de la librería)\n"
    f"🧹 genéricos: {len(CFG.genericos_todos)} "
    f"({len(GENERICOS_ESTRUCTURALES)} estructurales + {len(GENERICOS_SECTOR)} de sector "
    f"+ {len(GENERICOS_GEOGRAFIA)} topónimos + {len(GENERICOS_PROPIOS)} propios)\n"
    f"🧹 de ellos cuentan como ruido al comparar: {len(CFG.genericos_estructurales)}"
)

## 📥▶️ 6 y 7 · Lectura y ejecución

Se separan a propósito: leer, verificar el contrato, convertir las métricas, comprobar la
cobertura del catálogo de países y correr el smoke test cuesta segundos; la corrida completa cuesta minutos. Si una columna está mal
escrita o el FOB llegó con separador de miles, es mejor saberlo antes.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
#  📥 6 · LECTURA, CONTRATO, MÉTRICAS, COBERTURA DE PAÍSES Y SMOKE TEST
# ══════════════════════════════════════════════════════════════════════════
from record_linkage.flujo import preparar_insumo_local
from record_linkage.utils.almacenamiento import dir_trabajo_seguro

limpiar_estado("BASE", "RESULTADO")
CRONO = Cronometro()  # acompaña toda la corrida: lectura → ejecución → exportación

with CRONO.fase("0a · Localizar el insumo y copiarlo a disco local"):
    RUTA_ENTRADA_DRIVE = resolver_entrada(CARPETA_ENTRADA, ARCHIVO_ENTRADA)
    RUTA_ENTRADA = preparar_insumo_local(RUTA_ENTRADA_DRIVE, "/content/insumos")
    DIR_SALIDA_DRIVE = Path(CARPETA_SALIDA)
    DIR_TRABAJO = dir_trabajo_seguro(DIR_SALIDA_DRIVE, etiqueta="importadores")
    print(f"   📥 entrada (Drive) : {RUTA_ENTRADA_DRIVE.name}  ·  "
          f"{RUTA_ENTRADA_DRIVE.stat().st_size / 1024**2:,.1f} MB")
    print(f"   ⚡ entrada (local) : {RUTA_ENTRADA}")
    print(f"   🛠️  trabajo (local) : {DIR_TRABAJO}")
    print(f"   💾 salida  (Drive) : {DIR_SALIDA_DRIVE}")

with CRONO.fase("0b · Lectura del archivo"):
    BASE = leer_base(
        RUTA_ENTRADA, separador=SEPARADOR_CSV, encoding=ENCODING_CSV, hoja=HOJA_EXCEL
    )

with CRONO.fase("0c · Contrato, métricas y composición"):
    verificar_contrato(BASE, CFG)
    BASE = convertir_metricas(BASE, CFG.cols_metricas, tolerancia=TOLERANCIA_NO_NUMERICOS)
    STATS_ENTRADA = perfilar_entrada(BASE, CFG)

#  Sobre la base COMPLETA y antes del smoke test: una grafía sin clasificar es
#  una decisión suya (celda 3), no un fallo del pipeline. Si el modo es
#  "detener", esta línea se detiene con la lista y los remedios.
with CRONO.fase("0d · Cobertura del catálogo de países"):
    COBERTURA_PAISES = reportar_cobertura_paises(BASE, CFG)

if SMOKE_TEST:
    with CRONO.fase(f"0e · Smoke test sobre {min(FILAS_SMOKE_TEST, len(BASE)):,} filas"):
        print("   (corre el pipeline dos veces sobre la muestra: por fases y por fachada)")
        if not smoke_test(BASE, CFG, n=FILAS_SMOKE_TEST):
            raise RuntimeError(
                "Qué pasó: el smoke test falló sobre una muestra.\n"
                "Por qué importa: la corrida completa fallaría igual, minutos después.\n"
                "Qué hacer: lea el error de arriba antes de continuar."
            )

CRONO.imprimir_resumen("⏱️  TIEMPOS · preparación de la entrada")
BASE.head(5)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
#  ▶️ 7 · EJECUCIÓN  (con cronómetro por fase y avance en pantalla)
# ══════════════════════════════════════════════════════════════════════════
limpiar_estado("RESULTADO")

if USAR_FASES:
    RESULTADO = ejecutar_por_fases(BASE, CFG, CRONO)
else:
    with CRONO.fase("1-8 · Pipeline completo (fachada, sin detalle por fase)"):
        RESULTADO = deduplicar_importadores(BASE, CFG)

print(
    f"\n✅ {len(BASE):,} filas → {len(RESULTADO.golden):,} importadores consolidados\n"
    f"   {RESULTADO.n_candidatos:,} pares candidatos · "
    f"{RESULTADO.n_cortes_cohesion:,} cortes por cohesión · "
    f"emparejamiento {formato_duracion(RESULTADO.segundos)}"
)
CRONO.imprimir_resumen("⏱️  TIEMPOS · acumulado hasta la ejecución")
RESULTADO.correlativa.head(10)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
#  🔍 8 · CONTROL DE CALIDAD
# ══════════════════════════════════════════════════════════════════════════
_t_qa = time.perf_counter()
print("── ✅ INVARIANTES ────────────────────────────────────────────────────")
print(RESULTADO.invariantes.to_string(index=False))
if not RESULTADO.todo_ok:
    raise AssertionError(
        "Qué pasó: falló al menos una invariante.\n"
        "Por qué importa: el resultado NO es utilizable; alguna fila se perdió, "
        "se duplicó, se asignó por debajo del umbral declarado o mezcló dos países.\n"
        "Qué hacer: revise la fila marcada FALLA antes de usar nada de esta corrida."
    )

print("\n── 📊 MÉTRICAS DE LA CORRIDA ─────────────────────────────────────────")
print(RESULTADO.metricas.to_string(index=False))

print("\n── 📈 SENSIBILIDAD AL UMBRAL (cota superior de fusiones) ─────────────")
print(RESULTADO.extra["SENSIBILIDAD"].to_string(index=False))

print("\n── 🧱 RECALL DEL BLOQUEO (fuerza bruta) ──────────────────────────────")
recall = RESULTADO.extra["RECALL_BLOQUEO"]
print(
    recall.to_string(index=False)
    if len(recall)
    else "  (desactivado: CFG.paises_auditoria_bloqueo está vacío)"
)

if len(RESULTADO.sugerencias_pais):
    print("\n── 🌍 PAÍSES SIN CLASIFICAR (añádalos a la celda 4) ──────────────────")
    print(RESULTADO.sugerencias_pais.to_string(index=False))
    print("  La sugerencia NO se aplica sola: NARNIA se parece a ARMENIA con 0,85.")

print(f"\n── 👁️  MUESTRA PARA REVISIÓN MANUAL: {len(RESULTADO.muestra)} asignaciones ───────")
print("   Llene VEREDICTO_MANUAL con OK / ERROR / DUDOSO y calcule la precisión")
print("   por banda. Es la única cifra de precisión que no se está inventando.")
display(RESULTADO.muestra.head(20))

print("\n── ⚠️  LOS 15 GRUPOS MÁS GRANDES (donde vive el riesgo) ──────────────")
display(
    RESULTADO.golden.nlargest(15, "N_VARIANTES_NOMBRE")[
        ["RAZON_SOCIAL_FINAL", "PAIS_FINAL", "N_VARIANTES_NOMBRE", "SIM_MINIMA"]
    ]
)

CRONO.marcas["9 · Control de calidad (revisión en pantalla)"] = time.perf_counter() - _t_qa
print(f"\n⏱️  control de calidad: {formato_duracion(CRONO.marcas['9 · Control de calidad (revisión en pantalla)'])}")

## 💾 9 · Exportación a Drive

Orden: **Parquet primero** (barato, conserva tipos, es el checkpoint del resultado),
después el XLSX —que es lo lento y lo que puede reventar por tamaño—, después la
`metadata.json`, y al final la copia a Drive. Si la sesión muere durante el Excel, la
corrida ya está en disco.

La `metadata.json` deja por escrito qué produjo qué: sello de tiempo, versión del motor,
perfil de columnas, ruta y **SHA-256** del archivo de entrada, composición medida,
parámetros completos y métricas de la corrida. Es lo que hace reproducible un resultado
seis meses después.

In [ ]:
%%time
# ══════════════════════════════════════════════════════════════════════════
#  💾 9 · EXPORTACIÓN  (Parquet → XLSX → metadata → Drive)
# ══════════════════════════════════════════════════════════════════════════
_sello = pd.Timestamp.now(tz="America/Bogota")
NOMBRE_BASE = (
    f"deduplicacion_importadores_{_sello:%Y%m%d_%H%M}"
    if VERSIONAR_SALIDA
    else "deduplicacion_importadores"
)

#  La tabla TIEMPOS se congela aquí, así que cubre todas las fases MENOS la
#  exportación misma: un cronómetro no puede medirse a sí mismo antes de correr.
#  El total con exportación incluida se imprime al final de esta celda.
TABLA_TIEMPOS = CRONO.resumen()
TABLAS = {**RESULTADO.tablas(), "TIEMPOS": TABLA_TIEMPOS}

METADATA = {
    "timestamp": _sello.isoformat(),
    "notebook": "07_deduplicar_importadores_razon_social_pais",
    "motor": f"rues-linker {VERSION_EN_DRIVE}",
    "origen_paquete": str(RAIZ_PAQUETE),
    "arbol_paquete": INVENTARIO,
    "modulos_omitidos_por_extras": IMPORTACION["omitidos"],
    "perfil_entrada": PERFIL,
    "paises_sin_clasificar": {
        "modo": CFG.paises_sin_clasificar,
        "grafias": COBERTURA_PAISES.to_dict(orient="records"),
    },
    "archivo_entrada": str(RUTA_ENTRADA_DRIVE),
    "bytes_entrada": RUTA_ENTRADA_DRIVE.stat().st_size,
    "sha256_entrada": sha256_archivo(RUTA_ENTRADA),
    "composicion_entrada": STATS_ENTRADA,
    "parametros": {k: str(v) for k, v in sorted(vars(CFG).items())},
    "metricas": RESULTADO.metricas.to_dict(orient="records"),
    "invariantes_ok": bool(RESULTADO.todo_ok),
    "tiempos_por_fase": TABLA_TIEMPOS.to_dict(orient="records"),
    "segundos_emparejamiento": round(RESULTADO.segundos, 2),
    "pares_candidatos": int(RESULTADO.n_candidatos),
}

print(f"💾 escribiendo {NOMBRE_BASE} …")
with CRONO.fase("10 · Exportación (Parquet → XLSX → metadata → Drive)"):
    ARCHIVOS = exportar(TABLAS, DIR_TRABAJO, DIR_SALIDA_DRIVE, NOMBRE_BASE, METADATA)

print(f"\n📁 {len(ARCHIVOS)} archivos en {DIR_SALIDA_DRIVE}")
CRONO.imprimir_resumen("⏱️  TIEMPOS · CORRIDA COMPLETA")

---

# 📖 10 · Referencia

## 📑 Cómo leer el resultado

- **CORRELATIVA** — una fila por fila de entrada. `RAZON_SOCIAL_IMPORTADOR` + `PAIS_ESTANDAR` originales junto a `RAZON_SOCIAL_FINAL` + `PAIS_FINAL` asignados, con `SIM_AL_FINAL` (qué tan parecido es el nombre original a su nombre final, 0–1) y las banderas `CAMBIO_NOMBRE` / `CAMBIO_PAIS`. **Es el entregable.**
- **GOLDEN** — una fila por importador final: nombre, país, cuántas grafías absorbió, cohesión mínima y las métricas sumadas (`USD_FOB_TOTAL`, `NUM_REGISTROS`).
- **PAISES** — correlativa del país: cada grafía original con su canónico, el código ISO-3 y el método de asignación.
- **REVISION** — los grupos que merecen ojo humano: muchos nombres distintos, cohesión baja o sin nombre utilizable.
- **MUESTRA_REVISION** — muestra estratificada por banda de similitud, con una columna vacía para su veredicto.
- **METRICAS / INVARIANTES / SENSIBILIDAD / RECALL_BLOQUEO / PARAMETROS** — la corrida, auditable.
- **TIEMPOS** — segundos y porcentaje del total por fase. Cubre todo menos la exportación misma; el total con exportación se imprime en pantalla.
- **`*__METADATA.json`** — sello de tiempo, SHA-256 del insumo, inventario del árbol de Drive, perfil, parámetros, tiempos por fase y métricas.

Dos identificadores, a propósito:

- **`ID_IMPORTADOR`** respeta el país: la misma empresa en dos destinos son dos filas. Es la definición pedida (empalme por razón social **y** país).
- **`ID_EMPRESA_GLOBAL`** ignora el país, uniendo grupos cuyo nombre normalizado final coincide **exactamente**. Sirve para «¿cuántas empresas distintas hay?»; no hace emparejamiento difuso entre países.

## 🧾 Cambiar de fuente de datos

Un perfil nuevo en `PERFILES_ENTRADA` (celda 3) y `PERFIL` apuntando a él. Nada más:
la verificación del contrato, la conversión de métricas, el perfilado y la metadata leen
todos de `CFG`, que se construye a partir del perfil. No hay una segunda lista de nombres
de columna en el notebook.

## 🎚️ Cómo calibrar

1. Corra con los valores por defecto.
2. Abra `MUESTRA_REVISION` y **llene `VEREDICTO_MANUAL`**. Con 40 casos por banda ya se ve dónde se rompe. La precisión por banda es el único número que no se está inventando.
3. Si sobran fusiones en la banda baja → suba `umbral_nombre` (0,86 · 0,88) o suba `alfa_idf`.
4. Si faltan empalmes obvios → baje `umbral_nombre` a 0,82–0,80 y **vuelva a revisar la muestra**: en la base de referencia el tramo 0,80–0,84 cae a ~0,75 de precisión.
5. Si ve un grupo absorbido por una palabra genérica → añádala a `GENERICOS_PROPIOS` (celda 5). Es más barato y más auditable que mover el umbral.
6. Si ve un consolidador comiéndose a sus clientes (`"‹marca› ‹cliente final›"`) → suba `umbral_nombre`; con `max_diferencia_informativa = 0` solo ocurre cuando lo que sobra es ruido declarado.
7. Si ve empresas hermanas unidas (`BARRY CALLEBAUT USA` con `… CANADA`) → `geografia_es_ruido=False`, y acepte perder los empalmes de sucursal.
8. Si la celda 6 se detiene por **países sin clasificar** → en la celda 4: `GRAFIAS_ADICIONALES` para las que sí sean países, `NO_SON_PAISES` para las que no (`OTROS`, `NO DEFINIDO`). No las «arregle» eligiendo la sugerencia sin mirar: en la primera corrida real acertó 6 de 18 (`IRAN → IRLANDA`, `AFGANISTAN → ALBANIA`).
9. Cambie **un** parámetro a la vez y compare `METRICAS` antes y después. Dos cambios simultáneos no se pueden atribuir.

## 🔧 Si necesita tocar el motor

No lo parchee en una celda. El motor vive en `record_linkage` y se entrega con su banco de
pruebas; un cambio ahí se mide antes y después
(`python scripts/banco.py --comparar antes despues`). Los puntos de extensión declarados
son: ampliar `CATALOGO_PAISES` (celda 4), ampliar las listas de `matching.genericos`
(celda 5), y pasar su propio `Comparator` a `CampoSpec(comparador=...)` si necesita otra
medida de similitud.

## 🚫 Lo que este notebook NO hace

- **No inventa una verdad de campo.** Sin etiquetas humanas no hay precision/recall reales; lo que hay son cohesión, invariantes, recall del bloqueo (ese sí exacto) y una muestra lista para etiquetar.
- **No une empresas entre países.** Por diseño: el país es variable de empalme. `ID_EMPRESA_GLOBAL` solo une coincidencias exactas del nombre normalizado.
- **No resuelve grupos económicos.** `HOLCIM` y `HOLCIM COSTA RICA` se unen si están en el mismo país destino; `ACESCO ECUADOR` y `ACESCO INTERNACIONAL` son entidades jurídicas distintas y el notebook las trata como una sola. Si necesita la distinción, suba `umbral_nombre` y revise.
- **No decide qué hacer con los importadores sin nombre.** `"0"`, `"NO DISPONIBLE"`, `"A LA ORDEN"` y similares quedan cada uno en su propio grupo, marcados `sin_nombre_utilizable`; la celda 6 los cuenta. En `snowflake_v2` son el 31 % del FOB (`NO DISPONIBLE`: 21,7 %). Ignorarlos sesgaría cualquier ranking; fusionarlos inventaría un importador gigante. `TO ORDER OF ‹banco›` nombra a la parte y se conserva como nombre: pendiente decidir si eso es lo que se quiere.
- **No adivina un país.** Una grafía que el catálogo no reconoce se detiene, o se aísla si usted la declaró en `NO_SON_PAISES`; nunca se asigna al país «más parecido».
- **No corrige el origen.** Si la consulta trae parejas `(nombre, país)` repetidas o FOB no numérico, el notebook lo detecta y se detiene, pero el arreglo es en Snowflake.

## 📐 Precisión medida sobre la base de referencia

160 asignaciones revisadas a mano (40 por banda), sobre 211.949 registros de
destinatarios de exportación colombianos —**la base anterior, no esta**—. «Estricta»
cuenta como error los casos dudosos; «amplia» los cuenta como aciertos.

- Banda 0,84–0,88 · 6.915 asignaciones · precisión estricta 0,825 · amplia 0,925
- Banda 0,88–0,92 · 4.223 asignaciones · precisión estricta 0,950 · amplia 1,000
- Banda 0,92–0,96 · 3.515 asignaciones · precisión estricta 0,850 · amplia 0,900
- Banda 0,96–1,00 · 41.952 asignaciones · precisión estricta 1,000 · amplia 1,000
- **Ponderada · 56.605 asignaciones · precisión estricta 0,966 · amplia 0,985**

**Recall del bloqueo: PC = 1,000** en las tres particiones auditadas por fuerza bruta
(GTM, NLD, DEU), con reducción del espacio de pares de 94–97 %.

> ⚠️ Estas cifras describen la base de destinatarios DIAN. La base `snowflake_v2` cubre
> 2016-01 a 2026-07 y puede tener otra composición de nombres. **Vuelva a etiquetar
> `MUESTRA_REVISION` en la primera corrida con la fuente nueva**; hasta entonces la
> precisión de esta base es desconocida, no 0,966.

Dónde está el error que queda, en orden de frecuencia:

1. **Empresas hermanas del mismo grupo** — `BARRY CALLEBAUT USA` con `BARRY CALLEBAUT CANADA`, `LATAM AIRLINES GROUP` con `LATAM AIRLINES ECUADOR`. Consecuencia directa de `geografia_es_ruido = True`.
2. **Consolidadores de carga** — `X PRESS SHAPEWEAR LL ‹cliente›` cae en el grupo de la marca. El importador declarado es el consolidador, así que el resultado es defendible, pero si le importa el cliente final, súbale el umbral y revise `REVISION`.
3. **Nombres degenerados** — razones sociales que tras normalizar quedan en una sola palabra corriente. Marcados en `REVISION`.

## 🔬 Cómo se hizo la medición que respalda los valores por defecto

- **Recall del bloqueo:** fuerza bruta sobre 5 particiones completas (GTM, NLD, CHL, DEU, CRI; 1.683 a 3.298 nombres). Se comparan **todos** los pares y se mide cuántos recupera el LSH.
- **Precisión:** muestra estratificada por banda de similitud, revisada caso por caso. La configuración por defecto se cambió **tres veces** por lo que mostró esa revisión.
- **Invariantes:** diez, verificadas en cada corrida; **detienen** el notebook si fallan. La décima («ningún grupo mezcla dos grafías de país») se añadió al medir que, sin ella, tres grafías sin clasificar compartían importador.

Referencias del método: Fellegi & Sunter (1969), *A Theory for Record Linkage*, JASA 64(328);
Spärck Jones (1972), *A statistical interpretation of term specificity*, J. Documentation 28(1);
Winkler (1990), *String comparator metrics and enhanced decision rules in the Fellegi-Sunter model*;
Christen (2012), *Data Matching*, Springer.